In [1]:
import re
from collections import defaultdict

# 1. Define transformation rules based on current preparation method
PREP_TRANSFORMATIONS = {
    "steamed": {
        "new_method": "Oven-Roasted",
        "details": "High-heat oven roast with olive oil, sea salt, and black pepper to caramelize sugars and eliminate waterlogged sogginess."
    },
    "boiled": {
        "new_method": "Oven-Roasted",
        "details": "High-heat oven roast with olive oil, sea salt, and black pepper to replace bland, water-soaked texture with crisp edges."
    },
    "raw": {
        "new_method": "Lightly Marinated & Flash-Grilled",
        "details": "Flash-grill or toss with extra virgin olive oil, lemon spritz, sea salt, and black pepper to cut bitterness and soften tough fibers."
    },
    "sauteed": {
        "new_method": "High-Heat Dry-Sear",
        "details": "Dry-sear on a hot flat-top with minimal olive oil, sea salt, and black pepper to prevent oil pooling and soggy stems."
    },
    "fried": {
        "new_method": "High-Heat Oven-Baked",
        "details": "Brush lightly with olive oil, sea salt, and black pepper, then bake at high heat to achieve crispness without heavy oil saturation."
    },
    "roasted": {
        "new_method": "Subtle Acid & Herb Finish",
        "details": "Maintain high-heat roasting, but finish post-cook with a light lemon spritz, olive oil glaze, coarse sea salt, and fresh parsley."
    }
}

def detect_cooking_method(food_name):
    """Detects the current cooking method from the food item name."""
    name_lower = food_name.lower()
    
    for method in PREP_TRANSFORMATIONS.keys():
        if method in name_lower:
            return method
        
    # Default fallback if no method keyword is explicitly present
    if any(veg in name_lower for veg in ["broccoli", "carrot", "green bean", "cauliflower", "tofu"]):
        return "steamed"  # Assume steam/boiled as standard default for plain veggies
    
    return "unknown"


def map_recommendation(food_name):
    """Generates dynamic recommendations based on item type and detected cooking method."""
    name_lower = food_name.lower()

    # Discrete Units & Bakery (Portion Downscale Rule)
    if any(k in name_lower for k in ["pizza", "cookie", "pie", "muffin", "donut", "brownie"]):
        return {
            "action_type": "~25% Unit Downscale",
            "details": "Reduce default unit mass by ~25% (e.g., 8-cut pie to 10-cut pie, or 50g to 37.5g cookie). Retain identical recipe."
        }

    # Sandwiches & Burgers (Bread Downscale Rule)
    elif any(k in name_lower for k in ["burger", "sandwich", "wrap", "macaroni", "lasagna"]):
        return {
            "action_type": "~25% Downscale / Unbundling",
            "details": "Downscale bun/bread mass by ~25% or offer a patty bowl format; shift pan-cut grid from 12-cut to 16-cut."
        }

    # Vegetables, Plant Proteins, and Grains (Dynamic Prep Swap)
    else:
        current_method = detect_cooking_method(food_name)
        
        if current_method in PREP_TRANSFORMATIONS:
            transform = PREP_TRANSFORMATIONS[current_method]
            return {
                "action_type": f"Prep Swap ({current_method.title()} -> {transform['new_method']})",
                "details": transform["details"]
            }
        
        return {
            "action_type": "Prep Upgrade",
            "details": "Toss with a light olive oil glaze, sea salt, and black pepper, or audit portion sizing by ~25%."
        }


def analyze_food_waste(waste_entries, served_data, threshold=0.20):
    """Sums waste, computes ratios, and provides method-aware recommendations."""
    total_wasted = defaultdict(float)
    for food_name, amount in waste_entries:
        total_wasted[food_name] += amount

    flagged_items = []
    for food_name, wasted_amount in total_wasted.items():
        total_served = served_data.get(food_name, 0.0)
        
        if total_served <= 0:
            continue
        
        waste_ratio = wasted_amount / total_served
        
        if waste_ratio > threshold:
            rec = map_recommendation(food_name)
            flagged_items.append({
                "food_name": food_name,
                "detected_method": detect_cooking_method(food_name).title(),
                "waste_ratio": f"{waste_ratio:.1%}",
                "total_wasted": round(wasted_amount, 2),
                "total_served": round(total_served, 2),
                "action_type": rec["action_type"],
                "recommendation": rec["details"]
            })

    flagged_items.sort(key=lambda x: float(x["waste_ratio"].strip('%')), reverse=True)
    return flagged_items

In [2]:
waste_logs = [
    ["Steamed Broccoli", 120],
    ["Raw Broccoli", 110],
    ["Sauteed Broccoli", 100],
    ["Roasted Broccoli", 90],
    ["Pepperoni Pizza", 140]
]

served_data = {
    "Steamed Broccoli": 300.0,   # R = 40.0%
    "Raw Broccoli": 300.0,       # R = 36.7%
    "Sauteed Broccoli": 300.0,   # R = 33.3%
    "Roasted Broccoli": 300.0,   # R = 30.0%
    "Pepperoni Pizza": 500.0     # R = 28.0%
}

results = analyze_food_waste(waste_logs, served_data, threshold=0.20)

for entry in results:
    print(f"Item: {entry['food_name']} (Detected Prep: {entry['detected_method']})")
    print(f"  Waste Ratio: {entry['waste_ratio']}")
    print(f"  Action: {entry['action_type']}")
    print(f"  Recommendation: {entry['recommendation']}\n")

Item: Steamed Broccoli (Detected Prep: Steamed)
  Waste Ratio: 40.0%
  Action: Prep Swap (Steamed -> Oven-Roasted)
  Recommendation: High-heat oven roast with olive oil, sea salt, and black pepper to caramelize sugars and eliminate waterlogged sogginess.

Item: Raw Broccoli (Detected Prep: Raw)
  Waste Ratio: 36.7%
  Action: Prep Swap (Raw -> Lightly Marinated & Flash-Grilled)
  Recommendation: Flash-grill or toss with extra virgin olive oil, lemon spritz, sea salt, and black pepper to cut bitterness and soften tough fibers.

Item: Sauteed Broccoli (Detected Prep: Sauteed)
  Waste Ratio: 33.3%
  Action: Prep Swap (Sauteed -> High-Heat Dry-Sear)
  Recommendation: Dry-sear on a hot flat-top with minimal olive oil, sea salt, and black pepper to prevent oil pooling and soggy stems.

Item: Roasted Broccoli (Detected Prep: Roasted)
  Waste Ratio: 30.0%
  Action: Prep Swap (Roasted -> Subtle Acid & Herb Finish)
  Recommendation: Maintain high-heat roasting, but finish post-cook with a light l

In [42]:
import json
from collections import defaultdict

# Safely check if the ollama python package is installed
try:
    import ollama
    OLLAMA_INSTALLED = True
except ImportError:
    OLLAMA_INSTALLED = False


SYSTEM_PROMPT = """
You are an expert AI culinary director for a university dining hall waste-reduction recommender system.
Your job is to analyze high-waste menu items and output a SINGLE, complete, actionable sentence detailing a customized kitchen recommendation. 
You will be provided with the Food Item and its exact Waste Ratio.

CORE OPERATIONAL RULES:
You have full creative freedom to determine the most reasonable, appetizing, or healthier preparation method to fix the root cause of the waste. You may suggest pivoting to a completely different cooking technique, flavor profile, or presentation style as long as it is culinary sound and practical for a dining hall.
Write exactly one clear, natural sentence containing either the recommended new cooking method or by how much the portion size should be shrunk. Do not include category names, labels, or dictionary fields in the text. Do NOT mention ANYTHING about waste or reducing waste in your recommendation sentence.

15-CATEGORY BASELINE STRATEGIES (Adapt creatively based on the specific food item):
1. SLICED ITEMS (PIES, PIZZA, BAKERY BARS) -> Slightly decrease the serving size by INCREASING the number of slices per pizza or number of baked goods per tray by a REASONABLE AMOUNT (for example, 8 to 10 or 10 to 12 or 4 to 5, go based off standard slices per item).
2. COOKED VEGETABLES -> When the waste ratio is greater than 0.3, recommend changing the current vegetable to a DIFFERENT vegetable with a similar nutritional profile and a different method of cooking. Pivot to a more appealing, flavorful, or healthier preparation method (e.g., choose an alternative to steaming/boiling) that improves texture and reduces soggy and/or bland waste. 
3. PLAIN GRAINS & STARCHES -> When the waste ratio is greater than 0.3, recommend changing the current grain to a DIFFERENT grain with a similar nutritional profile. Suggest different flavor profiles for the grains by adding in herbs and/or spices.
4. SANDWICHES, BURGERS & WRAPS -> Reduce bread bulk dynamically based on waste data, or suggest unbundled, healthier formats (like bowls or lettuce wraps).
5. CASSEROLES & PAN DISHES -> Adjust pan grid scoring dynamically based on the waste ratio to yield appropriately sized default portions.
6. WHOLE ROASTED MEATS & POULTRY -> Recommend carving, slicing, or resting techniques that improve portion control and retain moisture.
7. PLANT-BASED PROTEINS -> Suggest texture-enhancing cooking methods that make the protein more appetizing and less prone to degrading on a hot line.
8. BREAKFAST MAINS -> Scale down sizes per the waste ratio, or shift to fresher batch-cooking methods.
9. SOUPS, STEWS & CHILIS -> Recommend finishing techniques that brighten the flavor profile and prevent the dish from tasting flat or heavy.
10. FRUITS -> Suggest better cutting, portioning, or refreshing techniques to keep fruit looking vibrant and easy to eat.
11. RAW SALAD BAR VEGETABLES -> Recommend light preparations (like mild marination or specific cuts) to make raw items more palatable.
12. PASTA & NOODLE BAR -> Choose methods that prevent starch sticking and steam-table clumping.
13. SAUCES & DRESSINGS -> Pivot from heavy, cloying bulk options to lighter, more vibrant, or healthier flavor profiles.
14. SELF-SERVE BEVERAGES -> Suggest natural, fresh enhancements over artificial flavorings.
15. BAKERY ITEMS -> Scale down the size a little bit because people are more likely to finish an item if only a tiny bit remains. Do not change the recipe.

OUTPUT FORMAT:
Return ONLY a valid JSON object with a single key "recommendation" containing your sentence. 
"""

def generate_rule_based_recommendation(food_name, waste_ratio):
    """Fallback rule-based recommender outputting single, natural sentence recommendations."""
    name_lower = food_name.lower()

    # Discrete Units & Bakery
    if "pizza" in name_lower:
        rec = f"Reduce the slice size of {food_name} by ~25% by shifting from an 8-cut pie to a 10-cut pie while retaining the identical crust and topping recipe."
    elif "cookie" in name_lower:
        rec = f"Reduce the unit dough mass of {food_name} by ~25% (scaling portions from 50g down to ~37.5g per cookie) while keeping the MBakery recipe identical."
    elif any(k in name_lower for k in ["pie", "muffin", "donut", "brownie", "cake"]):
        rec = f"Pre-cut {food_name} into ~25% smaller default portions while retaining 100% of the original recipe."

    # Sandwiches, Burgers & Casseroles
    elif "burger" in name_lower:
        rec = f"Downscale the bun mass for {food_name} by ~25% or offer a custom patty bowl on a bed of fresh greens to eliminate discarded bread."
    elif any(k in name_lower for k in ["sandwich", "wrap"]):
        rec = f"Reduce the bread or wrap mass for {food_name} by ~25% to match actual student appetite."
    elif any(k in name_lower for k in ["macaroni", "lasagna", "casserole", "enchilada"]):
        rec = f"Reduce default portion bulk of {food_name} by ~25% by shifting pan grid scoring from 12 cuts to 16 cuts per pan."

    # Grains, Rice & Noodles
    elif any(k in name_lower for k in ["rice", "quinoa", "couscous", "pasta", "penne", "noodle", "barley"]):
        if waste_ratio > 0.3:
            rec = f"Pivot from {food_name} to a different grain of similar nutritional profile, utilizing herbs and spices rather than serving it plain."
        else:
            rec = f"Fluff {food_name} post-cook with extra virgin olive oil, sea salt, black pepper, and fresh chives to prevent clumping and sogginess."

    # Soups & Liquids
    elif any(k in name_lower for k in ["soup", "stew", "chili", "chowder"]):
        rec = f"Finish {food_name} before service with a fresh lemon spritz, olive oil glaze, sea salt, and fresh herbs to brighten flavor."

    # Vegetables & Proteins (Cooking Method Transformations)
    else:
        is_veg = any(v in name_lower for v in ["broccoli", "spinach", "carrot", "vegetable", "corn", "bean", "pea"])
        
        if is_veg and waste_ratio > 0.3:
            rec = f"Replace {food_name} with an alternative vegetable with a similar nutritional profile, utilizing high-heat roasting instead of steam-table holding."
        elif "steamed" in name_lower or "steam" in name_lower:
            rec = f"Shift {food_name} from steam table holding to high-heat oven roasting tossed with olive oil, sea salt, and black pepper to eliminate sogginess."
        elif "boiled" in name_lower or "boil" in name_lower:
            rec = f"Transition {food_name} from boiling to high-heat oven roasting with olive oil, sea salt, and black pepper to build caramelized edges."
        elif "raw" in name_lower:
            rec = f"Toss raw {food_name} with extra virgin olive oil, fresh lemon juice, sea salt, and black pepper to cut bitterness and enhance texture."
        elif "sauteed" in name_lower:
            rec = f"Dry-sear {food_name} on a hot flat-top grill with minimal olive oil, sea salt, and black pepper to prevent excess oil pooling."
        elif "roasted" in name_lower:
            rec = f"Maintain oven roasting for {food_name}, but finish post-cook with a light lemon spritz, extra virgin olive oil glaze, sea salt, and fresh parsley."
        else:
            rec = f"Toss {food_name} with a light extra virgin olive oil glaze, sea salt, and black pepper to elevate natural flavor."

    return {
        "recommendation": rec,
        "source": "Rule-Based Fallback"
    }


def generate_recommendation(food_name, waste_ratio, model="qwen2.5"):
    """Queries Ollama for a single-sentence recommendation, with graceful rule fallback."""
    if OLLAMA_INSTALLED:
        name_lower = food_name.lower()
        is_veg = any(v in name_lower for v in ["broccoli", "spinach", "carrot", "vegetable", "corn", "bean", "pea", "mix"])
        is_grain = any(k in name_lower for k in ["rice", "quinoa", "couscous", "pasta", "penne", "noodle", "barley"])
        
        # Base user message
        user_content = f"High-waste food item: '{food_name}'. Exact Waste Ratio: {waste_ratio:.2f}."
        
        # Aggressive prompt injection for >0.3 edge cases
        if waste_ratio > 0.3:
            if is_veg:
                user_content += " CRITICAL RULE 2 TRIGGERED: You MUST recommend replacing this with a COMPLETELY DIFFERENT vegetable (e.g., zucchini, kale, green beans) and a new cooking method. You are FORBIDDEN from keeping the current vegetable."
            elif is_grain:
                user_content += " CRITICAL RULE 3 TRIGGERED: You MUST recommend replacing this with a COMPLETELY DIFFERENT grain (e.g., farro, brown rice, bulgur) and new spices. You are FORBIDDEN from keeping the current grain."

        try:
            response = ollama.chat(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content}
                ],
                format="json"
            )
            data = json.loads(response["message"]["content"])
            return {
                "recommendation": data.get("recommendation", ""),
                "source": f"Ollama AI ({model})"
            }
        except Exception:
            pass

    return generate_rule_based_recommendation(food_name, waste_ratio)


def analyze_food_waste(waste_entries, served_data, threshold=0.20, model="llama3.2"):
    """Evaluates Waste Ratio R = Wasted / Served and outputs single-sentence recommendations."""
    total_wasted = defaultdict(float)
    for food_name, amount in waste_entries:
        total_wasted[food_name] += amount

    flagged_items = []
    for food_name, wasted_amount in total_wasted.items():
        total_served = served_data.get(food_name, 0.0)
        
        if total_served <= 0:
            continue
        
        waste_ratio = wasted_amount / total_served
        
        if waste_ratio > threshold:
            rec = generate_recommendation(food_name, waste_ratio, model=model)
            
            flagged_items.append({
                "food_name": food_name,
                "waste_ratio": waste_ratio,
                "formatted_ratio": f"{waste_ratio:.1%}",
                "total_wasted": round(wasted_amount, 2),
                "total_served": round(total_served, 2),
                "recommendation": rec.get("recommendation", ""),
                "engine_source": rec.get("source", "Unknown")
            })

    flagged_items.sort(key=lambda x: x["waste_ratio"], reverse=True)
    return flagged_items


# --- DEMONSTRATION RUN ---
if __name__ == "__main__":
    waste_logs = [
        ["Pepperoni Pizza", 140],
        ["Chocolate Chip Cookie", 120],
        ["Steamed Broccoli", 125],
        ["Turkey Burger", 90],
        ["Plain Quinoa", 60],
        ["Steamed Spinach", 60],
        ["Steamed Barley", 50],
        ["Salad Bar Spring Mix", 150],
        ["Mango Cheesecake", 30]
    ]

    served_data = {
        "Pepperoni Pizza": 500.0,         # R = 28.0%
        "Chocolate Chip Cookie": 400.0,   # R = 30.0%
        "Steamed Broccoli": 300.0,        # R = 41.7%
        "Turkey Burger": 350.0,           # R = 25.7%
        "Plain Quinoa": 250.0,            # R = 24.0%
        "Steamed Spinach": 90,            # R = 66.7%
        "Steamed Barley": 80,
        "Salad Bar Spring Mix": 500,
        "Mango Cheesecake": 300
    }

    results = analyze_food_waste(waste_logs, served_data, threshold=0.20)

    for entry in results:
        print(f"Item: {entry['food_name']} (Waste Ratio: {entry['formatted_ratio']})")
        print(f"Recommendation: {entry['recommendation']}\n")

Item: Steamed Spinach (Waste Ratio: 66.7%)
Recommendation: Replace steamed spinach with sautéed and briefly blanched green beans, and adjust the dish to incorporate a citrus-ginger dressing to enhance flavor and texture.

Item: Steamed Barley (Waste Ratio: 62.5%)
Recommendation: Replace steamed barley with a different grain like bulgur, with a Mediterranean-inspired flavor profile featuring cumin, paprika, and a squeeze of fresh lemon juice.

Item: Steamed Broccoli (Waste Ratio: 41.7%)
Recommendation: Consider replacing steamed broccoli with roasted green beans, which can be seasoned with a drizzle of olive oil, salt, and pepper to enhance flavor and texture.

Item: Chocolate Chip Cookie (Waste Ratio: 30.0%)
Recommendation: Decrease the serving size of the Chocolate Chip Cookie by 15-20% to achieve a more balanced and appetizing portion size.

Item: Salad Bar Spring Mix (Waste Ratio: 30.0%)
Recommendation: Recommend pivoting to a hand-leafed, cold-pressed, and lightly dressed spring mi

In [47]:
waste_logs = [
        ["Pepperoni Pizza", 14000],
        ["Chocolate Chip Cookie", 120000],
        ["Steamed Broccoli", 125],
        ["Turkey Burger", 90],
        ["Plain Quinoa", 60],
        ["Steamed Spinach", 60],
        ["Steamed Barley", 50],
        ["Salad Bar Spring Mix", 150],
        ["Mango Cheesecake", 30]
    ]

served_data = {
        "Pepperoni Pizza": 0,         # R = 28.0%
        "Chocolate Chip Cookie": 4000.0,   # R = 30.0%
        "Steamed Broccoli": 3.0,        # R = 41.7%
        "Turkey Burger": 350.0,           # R = 25.7%
        "Plain Quinoa": 0.0,            # R = 24.0%
        "Steamed Spinach": 1000,            # R = 66.7%
        "Steamed Barley": 80,
        "Salad Bar Spring Mix": 1000,
        "Mango Cheesecake": 300
    }
results = analyze_food_waste(waste_logs, served_data, threshold=0.20)

for entry in results:
    print(f"Item: {entry['food_name']} (Waste Ratio: {entry['formatted_ratio']})")
    print(f"Recommendation: {entry['recommendation']}\n")


Item: Steamed Broccoli (Waste Ratio: 4166.7%)
Recommendation: Replace steamed broccoli with roasted green beans to reduce bulk and enhance flavor, adjusting cooking time to maintain optimal crunch and texture.

Item: Chocolate Chip Cookie (Waste Ratio: 3000.0%)
Recommendation: Consider decreasing the serving size of individual cookies from 100g to 80g to optimize consumption and reduce leftovers.

Item: Steamed Barley (Waste Ratio: 62.5%)
Recommendation: Replace steamed barley with roasted quinoa and add a blend of cumin, coriander, and paprika to enhance flavor and texture.

Item: Turkey Burger (Waste Ratio: 25.7%)
Recommendation: Reduce the serving size of the Turkey Burger by increasing the number of patties per sandwich from 2 to 2.25, resulting in smaller, more manageable portions.

